<a href="https://colab.research.google.com/github/Nav-del/Machine-Learning-Course/blob/main/Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Random Forest Classification**

In [1]:
#Importing libraries
import pandas as pd
import numpy as np
import seaborn as sns   #To load the dataset

In [2]:
df = sns.load_dataset('penguins')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [4]:
df.shape

(344, 7)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [6]:
#to find number of Null values
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,2
bill_depth_mm,2
flipper_length_mm,2
body_mass_g,2
sex,11


In [8]:
#Drop the null values
df.dropna(inplace=True)

In [9]:
#Check the dataset for null values again
df.isnull().sum()

,0
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,0


no null values

**Feature Engineering**

In [11]:
#One hot encoding
#Convert categorical data into numeric
df.sex.unique()

array(['Male', 'Female'], dtype=object)

we use one hot encoding here when number of features is low otherwise it has many entries.

In [12]:
pd.get_dummies(df['sex']).head()

,Female,Male
0,False,True
1,True,False
2,True,False
4,True,False
5,False,True


In [13]:
sex= pd.get_dummies(df['sex'], drop_first=True)
sex.head()

,Male
0,True
1,False
2,False
4,False
5,True


dropped the first column as only 1 coulumn is enuought to tell weather MALE or FEMALE

In [14]:
#ONE HOT ENCODING to island feature
df.island.unique()

array(['Torgersen', 'Biscoe', 'Dream'], dtype=object)

In [15]:
pd.get_dummies(df['island']).head()

,Biscoe,Dream,Torgersen
0,False,False,True
1,False,False,True
2,False,False,True
4,False,False,True
5,False,False,True


In [16]:
island = pd.get_dummies(df['island'], drop_first=True)
island.head()

,Dream,Torgersen
0,False,True
1,False,True
2,False,True
4,False,True
5,False,True


In [17]:
#Concat the above dataframes (sex and island) to the main dataframe

In [18]:
new_data = pd.concat([df,island,sex], axis=1)

When using `pd.concat`:

*   `axis=0` (default): Concatenates DataFrames row by row, stacking them vertically.
*   `axis=1`: Concatenates DataFrames column by column, joining them horizontally. This is what was used in the `new_data = pd.concat([df,island,sex], axis=1)` line, effectively adding the new 'Dream', 'Torgersen', and 'Male' columns to the original `df`.

In [19]:
new_data.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,Dream,Torgersen,Male
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,False,True,True
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,False,True,False
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,False,True,False
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,False,True,False
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,False,True,True


see the last 3 frames, it has been added. First the df then the island and then the sex

In [20]:
#Drop the repeated columns
new_data.drop(['sex','island'],axis=1, inplace = True)

In [21]:
new_data.head()

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,Dream,Torgersen,Male
0,Adelie,39.1,18.7,181.0,3750.0,False,True,True
1,Adelie,39.5,17.4,186.0,3800.0,False,True,False
2,Adelie,40.3,18.0,195.0,3250.0,False,True,False
4,Adelie,36.7,19.3,193.0,3450.0,False,True,False
5,Adelie,39.3,20.6,190.0,3650.0,False,True,True


the required columns are deleted

Now we will use MAP function to convert categorical data to numeric

In [24]:
#Make new dataframe and add only species to that as we need to numeric conversion to Y
Y= new_data.species

In [25]:
Y.head()

,species
0,Adelie
1,Adelie
2,Adelie
4,Adelie
5,Adelie


In [27]:
Y.unique() #Find the unique elements

array(['Adelie', 'Chinstrap', 'Gentoo'], dtype=object)

In [28]:
Y = Y.map({'Adelie':0,'Chinstrap':1,'Gentoo':2})
Y.head()

,species
0,0
1,0
2,0
4,0
5,0


Now we drop the Species from new_data and add Y to the dataframe

In [29]:
new_data.drop('species',axis=1, inplace=True)

In [30]:
new_data.head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,Dream,Torgersen,Male
0,39.1,18.7,181.0,3750.0,False,True,True
1,39.5,17.4,186.0,3800.0,False,True,False
2,40.3,18.0,195.0,3250.0,False,True,False
4,36.7,19.3,193.0,3450.0,False,True,False
5,39.3,20.6,190.0,3650.0,False,True,True


In [31]:
X = new_data

Splitting the Data set into train and test data

In [32]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, random_state=0)

In [34]:
print("X_train: ",X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train:  (233, 7)
X_test:  (100, 7)
y_train:  (233,)
y_test:  (100,)


Train Random Forest Classification on Training Set

In [37]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=5, criterion='entropy', random_state=0)
classifier.fit(X_train, y_train)
#n_estimator=5 : 5 decision trees
#random_state=0 : code reusability
#here the critetion we have used is Entropy

RandomForestClassifier(criterion='entropy', n_estimators=5, random_state=0)

Predicting the TEST results

In [38]:
y_pred = classifier.predict(X_test)
y_pred

array([0, 0, 2, 0, 0, 0, 1, 2, 2, 1, 2, 0, 0, 1, 0, 0, 2, 0, 1, 0, 0, 0,
       2, 2, 2, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 1, 0, 1, 0, 2, 2, 0, 0,
       0, 0, 0, 0, 2, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 2, 0, 1, 0, 2, 0, 0,
       2, 2, 1, 2, 2, 1, 2, 1, 0, 2, 0, 2, 0, 2, 1, 2, 2, 2, 1, 2, 1, 0,
       0, 2, 2, 0, 2, 0, 2, 0, 2, 0, 2, 2])

To check Accuracy : CONFUSION MATRIX

In [39]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [40]:
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[48  0  0]
 [ 2 14  0]
 [ 0  0 36]]


in this matrix , only 2 classes have been misclassified

In [41]:
accuracy_score(y_test,y_pred)

0.98

In [42]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98        48
           1       1.00      0.88      0.93        16
           2       1.00      1.00      1.00        36

    accuracy                           0.98       100
   macro avg       0.99      0.96      0.97       100
weighted avg       0.98      0.98      0.98       100



We used critetion Entropy before
Now trying with Gini

In [46]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=9, criterion='gini', random_state=0)
classifier.fit(X_train, y_train)
#we used Gini here
#increased number of trees too

RandomForestClassifier(n_estimators=9, random_state=0)

In [47]:
y_pred = classifier.predict(X_test)

In [48]:
accuracy_score(y_test, y_pred)

0.99

With more trees Accuracy increased

*****